# A06 · Preference Tuning & RL for LLMs

**Track:** Advanced / LLM engineering (lesson 6 of 10) · **Time:** ~3 h · **Mode:** runs fully OFFLINE (NumPy only, CPU, < 1 min); no API key needed
**Prereqs:** A05 (SFT, LoRA), B03 (logistic regression), B09 (backprop), basic probability · **You will build:** a Bradley-Terry reward model, a KL-regularised policy optimiser, DPO (loss, analytic gradient, training), a GRPO-style group-advantage trainer, and a reward-hacking experiment

---
## 🎯 Why this matters
SFT (A05) teaches a model to imitate *one* ideal answer. But for most real questions there are many acceptable answers and the product question is *which is better*: more helpful, more honest, safer, better formatted. Humans find it far easier to **compare** two answers than to **write** the perfect one. Preference tuning turns those comparisons into a training signal; reinforcement learning with **verifiable rewards** (does the code pass the tests? is the final number right?) does the same without humans in the loop and is behind much of the recent progress on reasoning models (A07).

**Real scenario.** *A support-assistant team* has a model that answers billing questions in four recognisable styles: terse, structured, verbose and sycophantic ("What a great question!"). Their annotators compare pairs of answers. The team must (1) turn comparisons into a reward, (2) shift the model toward preferred answers *without* drifting into gibberish, and (3) make sure the optimiser does not discover that annotators secretly like flattery. This lesson builds that whole pipeline on a toy small enough to verify every gradient.

## 🧭 Learning objectives
By the end you can:
1. Fit a **Bradley-Terry** reward model from pairwise preferences, and explain what it can and cannot identify.
2. State the **KL-regularised RLHF objective**, derive its closed-form optimum $\pi^*\propto\pi_{ref}e^{r/\beta}$, and optimise it with an exact policy gradient.
3. **Derive the DPO loss** from that optimum, implement it with an analytic gradient, and check it numerically.
4. Implement **GRPO-style group-relative advantages** on a verifiable-reward task and say what the group baseline buys.
5. Demonstrate **reward hacking** (Goodhart's law) and name the mitigations and their limits.

## 1 · Concept: where preference tuning sits

```
  pretrain ──▶ SFT (imitate demos) ──▶ preference tuning ──────────────▶ deployed model
  next-token     A05                    (a) RLHF: reward model + RL (PPO etc.)
                                        (b) DPO: skip the reward model, optimise preferences directly
                                        (c) RLVR/GRPO: reward = programmatic check (tests, exact answer)
```

| Method | Needs | Learns | Moving parts |
|---|---|---|---|
| **RLHF** | pairwise human preferences | reward model, then policy by RL | RM, policy, reference, (value net for PPO) |
| **DPO** | pairwise preferences | policy directly | policy + frozen reference |
| **GRPO / RLVR** | a *verifier* (checker) per prompt | policy by RL with group baseline | policy, reference (optional KL), sampler |

Notation: $x$ prompt, $y$ response, $\pi_\theta(y|x)$ the policy, $\pi_{ref}$ the frozen SFT model, $r(x,y)$ a scalar reward. To make *every* step checkable, our "LLM" is a **softmax policy over 4 answer styles** (no prompt dependence for the first sections): $\pi_\theta(k)=\mathrm{softmax}(\theta)_k$. Real policies factor over tokens, but every formula below is the same with $\log\pi(y)=\sum_t\log\pi(y_t|\cdot)$.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)
STYLES = ["terse", "structured", "verbose", "sycophantic"]
FEATS = ["correct", "clear_structure", "length", "flattery", "hedging"]
# Rows: measurable properties of each style (0..1). Synthetic, but shaped like a real audit of answers.
PHI = np.array([[0.70, 0.30, 0.10, 0.00, 0.20],     # terse
                [0.90, 0.90, 0.50, 0.10, 0.10],     # structured
                [0.85, 0.50, 1.00, 0.20, 0.30],     # verbose
                [0.50, 0.40, 0.80, 1.00, 0.00]])    # sycophantic
PI_REF = np.array([0.40, 0.30, 0.20, 0.10])         # the SFT model's habits
softmax = lambda z: np.exp(z - z.max()) / np.exp(z - z.max()).sum()
log_softmax = lambda z: z - z.max() - np.log(np.exp(z - z.max()).sum())
sigmoid = lambda z: 1 / (1 + np.exp(-z))
kl = lambda p, q: float((p * (np.log(p + 1e-300) - np.log(q + 1e-300))).sum())
print("\n".join(f"{s:<12}" + " ".join(f"{v:5.2f}" for v in row) for s, row in zip(STYLES, PHI)), "\n(columns: " + ", ".join(FEATS) + ")")

## 2 · Build it: the Bradley-Terry reward model
**Model.** Each response has a latent score $r(y)$. The probability a rater prefers $y_w$ over $y_l$ is

$$P(y_w \succ y_l)=\sigma\big(r(y_w)-r(y_l)\big),\qquad \sigma(z)=\frac1{1+e^{-z}}.$$

(This is exactly what you get if each rater's perceived utility is $r(y)$ plus independent logistic/Gumbel noise.) Given $N$ labelled pairs, the **negative log-likelihood** is

$$\mathcal{L}_{RM}=-\frac1N\sum_i\log\sigma\big(r(y_w^i)-r(y_l^i)\big).$$

With a linear reward on features, $r(y)=w^\top\phi(y)$, let $m_i=w^\top(\phi_w^i-\phi_l^i)$. Then $\partial\mathcal L/\partial w=-\frac1N\sum_i(1-\sigma(m_i))(\phi_w^i-\phi_l^i)$: each pair pushes $w$ along the feature difference, **weighted by how wrong the model currently is** ($1-\sigma(m)$). (In a real RM, $\phi$ is the LLM's final hidden state and $w$ a linear head, trained end-to-end.)

**Identifiability.** Only *differences* appear, so $r$ is determined up to an additive constant: absolute reward values are meaningless, which matters when you later use $r$ as an RL signal.

Our "annotators" secretly have a **bias**: they like long, flattering answers more than an expert would. We generate 3,000 comparisons from that latent utility, then fit a reward model.

In [ ]:
W_ANN  = np.array([2.0, 1.0,  1.0,  1.5, -0.5])    # what annotators actually reward (length and flattery included!)
W_TRUE = np.array([2.5, 1.0, -0.3, -1.5, -0.2])    # what an expert audit says is good (used later, never seen by the RM)
rg = np.random.default_rng(0)

def make_pairs(n, w, rng):
    Fa, Fb = rng.uniform(size=(n, 5)), rng.uniform(size=(n, 5))
    pa = sigmoid((Fa - Fb) @ w)                                  # Bradley-Terry probability that a beats b
    a_wins = rng.uniform(size=n) < pa
    Fw, Fl = np.where(a_wins[:, None], Fa, Fb), np.where(a_wins[:, None], Fb, Fa)
    return Fw, Fl

Fw, Fl = make_pairs(3000, W_ANN, rg); Fw_te, Fl_te = make_pairs(2000, W_ANN, rg)
D, D_te = Fw - Fl, Fw_te - Fl_te

def rm_loss(w, D): return float(-np.log(sigmoid(D @ w)).mean())
def rm_grad(w, D): return -((1 - sigmoid(D @ w))[:, None] * D).mean(axis=0)

# gradient check
w_t = rg.normal(size=5) * 0.5; num = np.zeros(5)
for j in range(5):
    e = np.zeros(5); e[j] = 1e-6; num[j] = (rm_loss(w_t + e, D) - rm_loss(w_t - e, D)) / 2e-6
assert np.allclose(num, rm_grad(w_t, D), atol=1e-7), (num, rm_grad(w_t, D))

w_hat = np.zeros(5); hist = []
for step in range(3000):
    w_hat -= 1.0 * rm_grad(w_hat, D)
    if step % 100 == 0: hist.append(rm_loss(w_hat, D_te))
acc = lambda w, D: float(((D @ w) > 0).mean())
print("true annotator w :", W_ANN); print("learned reward w :", w_hat)
print(f"held-out pairwise accuracy: learned {acc(w_hat, D_te):.3f}  | the TRUE annotator utility itself: {acc(W_ANN, D_te):.3f}   (ceiling: labels are noisy)")
assert np.corrcoef(w_hat, W_ANN)[0, 1] > 0.97 and np.abs(w_hat - W_ANN).max() < 0.5
assert acc(w_hat, D_te) > acc(W_ANN, D_te) - 0.02 and acc(w_hat, D_te) > 0.7
assert hist[-1] <= hist[0], "held-out NLL must improve"

The learned reward recovers the annotators' weights - **including their bias** toward length and flattery. This is the central uncomfortable fact: *a reward model learns what raters reward, not what you meant.* Hold that thought for section 7. Also notice the ceiling: even the true utility only gets ~70% accuracy because human preferences are noisy, so do not expect 99% RM accuracy.

## 3 · The RLHF objective, its closed-form optimum, and an exact policy gradient
Maximising reward alone is dangerous (section 7). RLHF instead maximises reward **minus a KL penalty** to the reference model:

$$J(\pi)=\mathbb{E}_{y\sim\pi}\big[r(y)\big]-\beta\,\mathrm{KL}\big(\pi\,\|\,\pi_{ref}\big)=\sum_y \pi(y)\Big[r(y)-\beta\log\frac{\pi(y)}{\pi_{ref}(y)}\Big].$$

**Closed form.** Maximise over distributions with a Lagrange multiplier $\lambda$ for $\sum\pi=1$: $\partial/\partial\pi(y)$ gives $r(y)-\beta\log\frac{\pi(y)}{\pi_{ref}(y)}-\beta-\lambda=0$, hence

$$\boxed{\pi^*(y)=\frac{1}{Z}\,\pi_{ref}(y)\,\exp\!\big(r(y)/\beta\big),\qquad Z=\sum_y\pi_{ref}(y)e^{r(y)/\beta}.}$$

So the optimum **re-weights the reference by exponentiated reward**: $\beta\to\infty$ gives $\pi_{ref}$ back; $\beta\to0$ concentrates on the argmax of $r$. In a real LLM the sum over $y$ is intractable (that is why we need RL or DPO), but in our 4-style world we can compute it exactly and *check the optimiser against it*.

**Policy gradient.** With $\pi=\mathrm{softmax}(\theta)$ let $g_k=\partial J/\partial\pi_k=r_k-\beta\log\frac{\pi_k}{\pi_{ref,k}}-\beta$. The softmax Jacobian is $\partial\pi_k/\partial\theta_j=\pi_k(\delta_{kj}-\pi_j)$, so $\ \partial J/\partial\theta_j=\pi_j\,(g_j-\sum_k\pi_kg_k)$. (This equals the REINFORCE form $\mathbb E_\pi[(\text{reward}_{\beta}-b)\nabla\log\pi]$ computed *exactly* instead of from samples.)

In [ ]:
R_PROXY = PHI @ w_hat                        # the learned reward model scores each style
print("proxy reward per style:", dict(zip(STYLES, R_PROXY.round(2))))

def J(theta, r, beta, pref=PI_REF):
    p = softmax(theta); return float((p * (r - beta * np.log(p / pref))).sum())

def J_grad(theta, r, beta, pref=PI_REF):
    p = softmax(theta); g = r - beta * np.log(p / pref) - beta
    return p * (g - (p * g).sum())

# numeric check of the analytic gradient
th = np.random.default_rng(1).normal(size=4)
num = np.array([(J(th + e, R_PROXY, 0.7) - J(th - e, R_PROXY, 0.7)) / 2e-6 for e in np.eye(4) * 1e-6])
assert np.allclose(num, J_grad(th, R_PROXY, 0.7), atol=1e-7)

def optimise(r, beta, steps=6000, lr=2.0, theta0=None):
    th = np.log(PI_REF).copy() if theta0 is None else theta0.copy(); traj = []
    for t in range(steps):
        th += lr * J_grad(th, r, beta); traj.append(th.copy())
    return th, traj

def pi_star(r, beta): w = PI_REF * np.exp((r - r.max()) / beta); return w / w.sum()

for beta in [2.0, 0.5, 0.25]:
    th, _ = optimise(R_PROXY, beta); p = softmax(th)
    print(f"beta={beta:<4} policy {p.round(3)}   closed-form {pi_star(R_PROXY, beta).round(3)}   KL(pi*||pi) = {kl(pi_star(R_PROXY, beta), p):.1e}")
    assert kl(pi_star(R_PROXY, beta), p) < 1e-5, "gradient ascent must converge to the closed-form optimum"

Gradient ascent lands on the closed-form $\pi^*$ for every $\beta$: the KL-regularised objective, the derivation and the code agree. Now the **trade-off frontier**: sweep $\beta$ and plot expected reward against drift from the reference. Every deployed RLHF run picks a point on a curve like this.

In [ ]:
betas = np.logspace(-1.3, 1.3, 25)
pts = [(kl(pi_star(R_PROXY, b), PI_REF), float(pi_star(R_PROXY, b) @ R_PROXY)) for b in betas]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(*zip(*pts), "o-"); ax[0].set_xlabel("KL(pi || pi_ref)  (drift)"); ax[0].set_ylabel("expected proxy reward"); ax[0].set_title("reward vs drift (each point = one beta)")
for j, b in enumerate([2.0, 0.5, 0.1]): ax[1].bar(np.arange(4) + 0.27 * (j - 1), pi_star(R_PROXY, b), width=0.27, label=f"beta={b}")
ax[1].set_xticks(range(4)); ax[1].set_xticklabels(STYLES); ax[1].legend(); ax[1].set_title("optimal policy for beta = 2, 0.5, 0.1"); plt.tight_layout(); plt.show()
ks, rs = zip(*pts); assert np.all(np.diff(rs) <= 1e-9) and np.all(np.diff(ks) <= 1e-9), "betas ascend: larger beta -> less reward and less drift"

### PPO in one paragraph (what runs at scale)
With a real LLM you cannot enumerate $y$, so you *sample* responses and use a policy-gradient method. **PPO** collects a batch from the current policy, estimates an advantage $A$ per token (reward minus a learned value baseline, smoothed with GAE), and maximises the **clipped surrogate** $\mathbb E\big[\min(\rho A,\ \mathrm{clip}(\rho,1-\epsilon,1+\epsilon)A)\big]$ where $\rho=\pi_\theta/\pi_{old}$, so one batch can be reused for several gradient steps without the policy jumping too far. The KL term appears either as a per-token reward penalty or a loss term, and a **value network** (as large as the policy) is trained alongside. That is four large models in memory (policy, reference, reward, value) - the main practical motivation for DPO and GRPO. The clipping behaviour is simple enough to verify:

In [ ]:
def ppo_clip_objective(rho, A, eps=0.2): return np.minimum(rho * A, np.clip(rho, 1 - eps, 1 + eps) * A)
rho = np.array([0.5, 0.9, 1.0, 1.1, 1.5])
print("A=+1:", ppo_clip_objective(rho, +1.0), "\nA=-1:", ppo_clip_objective(rho, -1.0))
assert np.isclose(ppo_clip_objective(rho, +1.0)[-1], 1.2) and np.isclose(ppo_clip_objective(rho, -1.0)[0], -0.8) and np.isclose(ppo_clip_objective(rho, -1.0)[-1], -1.5), "clip caps the upside for A>0 and keeps the pessimistic bound for A<0"
print("-> with A>0 gains stop at rho=1.2 (no incentive to move further); with A<0 the min keeps the larger penalty (at rho=1.5 the loss is -1.5, not clipped to -1.2).")

## 4 · DPO: skip the reward model
**Key observation.** Invert the closed form $\pi^*=\pi_{ref}e^{r/\beta}/Z$:

$$r(y)=\beta\log\frac{\pi^*(y)}{\pi_{ref}(y)}+\beta\log Z.$$

The reward is a (scaled) *log-ratio between the optimal policy and the reference*, plus a prompt-dependent constant $\beta\log Z$. Plug this into the Bradley-Terry likelihood: because BT only uses **differences** $r(y_w)-r(y_l)$, the intractable $\log Z$ **cancels**:

$$P(y_w\succ y_l)=\sigma\!\Big(\beta\log\frac{\pi^*(y_w)}{\pi_{ref}(y_w)}-\beta\log\frac{\pi^*(y_l)}{\pi_{ref}(y_l)}\Big).$$

Now *parametrise* the policy $\pi_\theta$ in place of $\pi^*$ and maximise the likelihood of the observed preferences:

$$\boxed{\mathcal L_{DPO}(\theta)=-\mathbb E_{(y_w,y_l)}\Big[\log\sigma\big(\beta\,[\Delta_w-\Delta_l]\big)\Big],\qquad \Delta_y=\log\pi_\theta(y)-\log\pi_{ref}(y).}$$

(Rafailov et al., 2023.) No reward model, no sampling loop, no value net: a supervised-style loss on pairs. **Gradient.** Let $m=\beta(\Delta_w-\Delta_l)$. Then

$$\nabla_\theta\mathcal L=-\beta\,\sigma(-m)\,\big[\nabla\log\pi_\theta(y_w)-\nabla\log\pi_\theta(y_l)\big].$$

Read it: raise the likelihood of the winner, lower that of the loser, **weighted by $\sigma(-m)$ = how badly the implicit reward currently ranks the pair**. For our softmax policy, $\nabla_\theta\log\pi_\theta(k)=e_k-\pi_\theta$, so the $\pi_\theta$ terms *cancel* in the difference and

$$\nabla_\theta\mathcal L=-\beta\,\sigma(-m)\,(e_{y_w}-e_{y_l}).$$

We implement the loss with real log-softmaxes (so it is the literal formula), the analytic gradient above, and check them against finite differences. Preference data: pairs of styles sampled from $\pi_{ref}$ (as in practice, where pairs come from the SFT model), labelled by the annotators' Bradley-Terry utility.

In [ ]:
R_ANN = PHI @ W_ANN                       # annotators' latent reward per style (used ONLY to generate labels)
rd = np.random.default_rng(5)
def sample_pairs(n, pref=PI_REF, allowed=None):
    pr = pref if allowed is None else pref * allowed / (pref * allowed).sum()
    a, b = rd.choice(4, n, p=pr), rd.choice(4, n, p=pr); keep = a != b; a, b = a[keep], b[keep]
    a_wins = rd.uniform(size=len(a)) < sigmoid(R_ANN[a] - R_ANN[b])
    return np.where(a_wins, a, b), np.where(a_wins, b, a)       # (winner idx, loser idx)

YW, YL = sample_pairs(20000)
print(f"{len(YW)} preference pairs; winner-style frequency: {np.bincount(YW, minlength=4) / len(YW)}")

def dpo_loss(theta, YW, YL, beta, pref=PI_REF):
    d = log_softmax(theta) - np.log(pref)                       # Delta_y for every style
    m = beta * (d[YW] - d[YL]); return float(-np.log(sigmoid(m)).mean())

def dpo_grad(theta, YW, YL, beta, pref=PI_REF):
    d = log_softmax(theta) - np.log(pref); m = beta * (d[YW] - d[YL]); w = beta * sigmoid(-m) / len(YW)
    g = np.zeros(4); np.add.at(g, YW, -w); np.add.at(g, YL, +w); return g          # -beta*sigma(-m)*(e_w - e_l), averaged

th = np.random.default_rng(2).normal(size=4)
num = np.array([(dpo_loss(th + e, YW, YL, 0.5) - dpo_loss(th - e, YW, YL, 0.5)) / 2e-6 for e in np.eye(4) * 1e-6])
print("analytic:", dpo_grad(th, YW, YL, 0.5), "\nnumeric :", num)
assert np.allclose(num, dpo_grad(th, YW, YL, 0.5), atol=1e-8)
assert abs(dpo_grad(th, YW, YL, 0.5).sum()) < 1e-12, "gradient sums to zero: logits are shift-invariant"
assert abs(dpo_loss(np.log(PI_REF), YW, YL, 0.5) - np.log(2)) < 1e-12, "at pi = pi_ref the margin is 0 so loss = log 2"

Two sanity facts verified above: at $\pi=\pi_{ref}$ every margin is zero so the loss is exactly $\log 2$, and the gradient sums to zero (softmax logits are shift-invariant). Now train and compare with the *true* optimum for the annotators' reward, $\pi^*\propto\pi_{ref}e^{R_{ann}/\beta}$ - a target DPO never sees directly.

In [ ]:
def train_dpo(YW, YL, beta, steps=3000, lr=2.0 ):
    th = np.log(PI_REF).copy(); losses = []
    for t in range(steps):
        th -= lr * dpo_grad(th, YW, YL, beta) / beta; losses.append(dpo_loss(th, YW, YL, beta))   # /beta only rescales the step size
    return th, losses

BETA = 1.0
th_dpo, losses = train_dpo(YW, YL, BETA); p_dpo = softmax(th_dpo)
target = pi_star(R_ANN, BETA)
print("DPO policy      :", p_dpo.round(3)); print("closed-form pi* :", target.round(3)); print(f"KL(pi*||DPO) = {kl(target, p_dpo):.4f}   (data noise, 20k pairs)")
th_rlhf, _ = optimise(R_PROXY, BETA); print("RLHF via learned RM:", softmax(th_rlhf).round(3), f" KL(DPO||RLHF) = {kl(p_dpo, softmax(th_rlhf)):.4f}")
impl = BETA * (log_softmax(th_dpo) - np.log(PI_REF))
print("implicit reward (centred):", (impl - impl.mean()).round(2), " true (centred):", (R_ANN - R_ANN.mean()).round(2))
assert losses[-1] < losses[0] < np.log(2) + 1e-9
assert kl(target, p_dpo) < 0.01, "DPO should approach the KL-regularised optimum for the annotators' reward"
assert kl(p_dpo, softmax(th_rlhf)) < 0.05, "two routes (RM+RL vs DPO) agree, because they fit the same preferences"
assert np.corrcoef(impl, R_ANN)[0, 1] > 0.99
plt.figure(figsize=(5.5, 3.3)); plt.plot(losses); plt.axhline(np.log(2), color="grey", ls=":"); plt.xlabel("step"); plt.ylabel("DPO loss"); plt.title("DPO training loss (dotted = log 2)"); plt.show()

**What this shows.** (i) DPO and RLHF-through-a-reward-model reach nearly the same policy, as theory predicts (same BT likelihood, same KL-regularised optimum). (ii) DPO's *implicit reward* $\beta\log\pi_\theta/\pi_{ref}$ recovers the annotators' rewards up to a constant. (iii) Because both are fit to *annotator* preferences, both inherit their bias - DPO is not "safer", just simpler.

## 5 · Use it: GRPO and verifiable rewards
For maths, code and structured extraction we do not need humans or a learned reward: a **verifier** returns 1 if the final answer is right (or tests pass), else 0. **GRPO** (Shao et al., 2024, introduced in the DeepSeekMath work) removes PPO's value network by using the *other samples for the same prompt* as the baseline. For a prompt $x$ sample a group of $G$ responses with rewards $r_1..r_G$ and set

$$A_i=\frac{r_i-\bar r}{\mathrm{std}(r)+\varepsilon},\qquad \bar r=\tfrac1G\sum_j r_j.$$

Every response in the group then gets the advantage of its own reward relative to its siblings, and the policy gradient is $\frac1G\sum_iA_i\nabla\log\pi_\theta(y_i)$ (plus a clipped ratio and a KL penalty to the reference in the full algorithm; with one update per batch the ratio is 1). Three properties to internalise:

* **It is a baseline**: subtracting $\bar r$ does not bias the gradient (to leading order; the leave-one-in mean makes a tiny bias for small $G$) but cuts variance a lot.
* **All-correct or all-wrong groups give zero advantage** (std = 0): *no learning signal*, so prompts must be at the right difficulty (not trivially easy, not hopeless).
* Dividing by std rescales per-prompt: prompts with success rates near 0 or 1 get larger-magnitude advantages for their rare outliers.

**Toy.** Three prompt types (easy / medium / hard), each with 5 solution *strategies* whose success probabilities are fixed numbers (synthetic). The policy keeps separate softmax logits per prompt type. Compare: REINFORCE with *no* baseline, with a mean baseline (no std), and GRPO (mean and std). Reward = 1 if the sampled strategy "solves" the problem.

In [ ]:
STRATS = ["guess", "short_cot", "long_cot", "cot+selfcheck", "cot+calculator"]
P_SOLVE = np.array([[0.50, 0.95, 0.97, 0.97, 0.99],      # easy
                    [0.20, 0.60, 0.75, 0.85, 0.90],      # medium
                    [0.02, 0.10, 0.25, 0.35, 0.50]])     # hard
def expected_success(TH): return float(np.mean([softmax(TH[i]) @ P_SOLVE[i] for i in range(3)]))

def run_rl(method, seed, steps=300, G=8, lr=0.5):
    rng = np.random.default_rng(seed); TH = np.zeros((3, 5)); curve = []; zero_groups = 0; total = 0
    for t in range(steps):
        i = rng.integers(3); p = softmax(TH[i]); y = rng.choice(5, G, p=p); r = (rng.uniform(size=G) < P_SOLVE[i, y]).astype(float)
        if method == "reinforce": A = r
        elif method == "mean_baseline": A = r - r.mean()
        elif method == "grpo": A = (r - r.mean()) / (r.std() + 1e-6)
        zero_groups += int(np.allclose(A, 0)) if method != "reinforce" else int(r.std() == 0); total += 1
        grad = np.zeros(5)
        for a, yi in zip(A, y): grad += a * (np.eye(5)[yi] - p)
        TH[i] += lr * grad / G; curve.append(expected_success(TH))
    return np.array(curve), zero_groups / total

# ---- sanity: a group with identical rewards has zero advantage (nothing to learn from)
r_same = np.ones(8); assert np.allclose((r_same - r_same.mean()) / (r_same.std() + 1e-6), 0)

curves = {m: [] for m in ["reinforce", "mean_baseline", "grpo"]}; zero_frac = {}
for m in curves:
    outs = [run_rl(m, s) for s in range(12)]; curves[m] = np.mean([o[0] for o in outs], axis=0); zero_frac[m] = np.mean([o[1] for o in outs])
print(f"initial expected success (uniform policy): {expected_success(np.zeros((3, 5))):.3f}")
for m, c in curves.items(): print(f"{m:<14} after 300 steps: {c[-1]:.3f}   (mean over 12 seeds)   groups with zero advantage: {zero_frac[m]:.0%}")
plt.figure(figsize=(6, 3.5))
for m, c in curves.items(): plt.plot(c, label=m)
plt.xlabel("training step (one prompt group each)"); plt.ylabel("expected solve rate"); plt.title("verifiable-reward RL on 3 prompt types"); plt.legend(); plt.show()
start = expected_success(np.zeros((3, 5)))
assert all(c[-1] > start + 0.1 for c in curves.values()), "all variants learn"
assert curves["grpo"][100] > curves["reinforce"][100] and curves["grpo"][100] > curves["mean_baseline"][100], "GRPO is fastest at this step size (see the note below on why)"

Read the curves carefully (all three reach similar plateaus; the ceiling is the average best-strategy solve rate, about 0.80). **GRPO is fastest** here, but mostly because dividing by the group std *enlarges* the step on prompts where successes are rare (hard prompt) - a scale effect, not a variance effect. The plain **mean baseline is not faster** than raw REINFORCE; its job is **variance reduction**, which matters more at larger learning rates and with noisier rewards. Measure that directly rather than argue: sample the gradient estimate many times at the initial (uniform) policy on the medium prompt and compare its mean and spread.

In [ ]:
def grad_sample(method, i, TH, rng, G=8):
    p = softmax(TH[i]); y = rng.choice(5, G, p=p); r = (rng.uniform(size=G) < P_SOLVE[i, y]).astype(float)
    A = {"reinforce": r, "mean_baseline": r - r.mean(), "grpo": (r - r.mean()) / (r.std() + 1e-6)}[method]
    return sum(a * (np.eye(5)[yi] - p) for a, yi in zip(A, y)) / G

rng = np.random.default_rng(11); TH0 = np.zeros((3, 5)); med = 1
stats = {}
for m in ["reinforce", "mean_baseline", "grpo"]:
    S = np.array([grad_sample(m, med, TH0, rng) for _ in range(4000)]); stats[m] = (S.mean(0), S.var(0).sum())
    print(f"{m:<14} mean grad {stats[m][0].round(3)}   total variance {stats[m][1]:.4f}")
true_dir = P_SOLVE[med] - softmax(TH0[med]) @ P_SOLVE[med]; true_dir = softmax(TH0[med]) * true_dir     # exact policy gradient of expected solve rate
print("exact gradient (up to scale):", true_dir.round(3))
assert stats["mean_baseline"][1] < stats["reinforce"][1], "the baseline cuts variance"
assert np.allclose(stats["reinforce"][0], true_dir, atol=0.01), "REINFORCE is an unbiased estimate of the exact policy gradient"
assert np.allclose(stats["mean_baseline"][0], true_dir * (7 / 8), atol=0.01), "mean baseline *including the sample itself* shrinks the signal by (G-1)/G = 7/8, not zero"
cos = lambda a, b: float(a @ b / np.linalg.norm(a) / np.linalg.norm(b))
assert cos(stats["grpo"][0], true_dir) > 0.95, "std-normalised advantage still points the right way"

Three measurements worth remembering: (1) REINFORCE matches the exact policy gradient on average (unbiased) but is noisy; (2) including each sample in its own mean baseline scales the expected gradient by exactly $(G-1)/G$ - a known small-group effect, harmless for direction; (3) std normalisation changes the *scale per prompt* but not the direction. In practice GRPO runs also need: a KL penalty to the reference, sampling temperature > 0 (to get diverse groups), and prompts filtered for non-trivial pass rates.

## 6 · Break it: DPO has no opinion about what it never saw
Suppose the preference dataset was collected only among styles `terse`, `structured`, `verbose` (the sycophantic style was never shown to annotators, e.g. because an older model produced the pairs). DPO's gradient is $-\beta\sigma(-m)(e_w-e_l)$: *the logit of an unseen style never receives gradient*. But probabilities are normalised, so its probability still changes as the others move.

In [ ]:
allowed = np.array([1, 1, 1, 0.0])
YWo, YLo = sample_pairs(20000, allowed=allowed)
assert 3 not in set(YWo) | set(YLo)
th_off, _ = train_dpo(YWo, YLo, BETA); p_off = softmax(th_off)
print("reference:", PI_REF, "\noff-support DPO:", p_off.round(3))
print("logit of unseen style changed by", float(abs(th_off[3] - np.log(PI_REF)[3])), "; probability went", PI_REF[3], "->", round(float(p_off[3]), 3))
assert th_off[3] == np.log(PI_REF)[3], "no gradient ever reached the unseen style's logit"
assert abs(p_off[3] - PI_REF[3]) > 0.005
print("Lesson: the unseen style's probability is whatever normalisation leaves behind. On-policy (or at least on-distribution) preference data matters; this is one reason iterative/online DPO and RL on fresh samples are used.")

## 7 · Break it: reward hacking (Goodhart's law)
*"When a measure becomes a target, it ceases to be a good measure."* Our reward model learned the annotators' bias ($w_{ann}$ rewards length and flattery). An **expert audit** ($w_{true}$) penalises flattery. We now *over-optimise the proxy* (learned RM, small $\beta$) and track the proxy reward, the true audit score, and the KL from the reference as optimisation proceeds. In real RLHF, the analogous "gold vs proxy reward" divergence under heavy optimisation is documented in Gao, Schulman & Hilton (2022), *Scaling Laws for Reward Model Overoptimization*; here it falls out of the toy directly.

In [ ]:
R_TRUE = PHI @ W_TRUE
print(f"{'style':<12}{'proxy (learned RM)':>20}{'true (expert audit)':>22}")
for s, a, b in zip(STYLES, R_PROXY, R_TRUE): print(f"{s:<12}{a:>20.2f}{b:>22.2f}")

th, traj = optimise(R_PROXY, beta=0.0, steps=1500, lr=0.15)         # beta = 0: pure reward maximisation
P = np.array([softmax(t) for t in traj])
proxy_c, true_c, kl_c = P @ R_PROXY, P @ R_TRUE, np.array([kl(p, PI_REF) for p in P])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(proxy_c, label="proxy reward (what we optimise)"); ax[0].plot(true_c, label="true audit score"); ax[0].set_xlabel("optimisation step"); ax[0].legend(); ax[0].set_title("proxy keeps rising, truth peaks then falls")
ax[1].plot(kl_c, color="C3"); ax[1].set_xlabel("optimisation step"); ax[1].set_ylabel("KL(pi || pi_ref)"); ax[1].set_title("drift from the reference"); plt.tight_layout(); plt.show()
peak = int(true_c.argmax())
print(f"true score peaks at step {peak} (KL={kl_c[peak]:.2f}, true={true_c[peak]:.2f}); at the end: true={true_c[-1]:.2f}, proxy={proxy_c[-1]:.2f}; final policy {P[-1].round(3)}")
assert np.all(np.diff(proxy_c) > -1e-12), "the optimiser improves the proxy monotonically"
assert true_c[peak] > true_c[0] + 0.1, "early on, optimising the proxy genuinely helps"
assert true_c[-1] < true_c[peak] - 0.5, "...then Goodhart: continued optimisation of the proxy destroys the true objective"
assert P[-1].argmax() == 3, "the optimum of the proxy is the sycophantic style"

**Mitigations, and what each can and cannot do** (test each on your own eval, none is a guarantee):

| Mitigation | Helps with | Does *not* fix |
|---|---|---|
| **KL penalty** ($\beta$) / early stopping on a gold eval | limits how far the policy exploits the proxy | a proxy that is wrong in the region you *do* reach |
| **Reward-model ensembles / uncertainty** (pessimistic reward) | noise and out-of-distribution over-confidence | *systematic bias shared by all members* (they all learned the same annotator bias) |
| **Better preference data** (balance length/flattery, adversarial pairs, expert audit slices) | removes the bias at its source | cost |
| **Verifiable rewards** (tests, exact answers) | no learned proxy to hack | the *verifier* can still be gamed (e.g. special-casing the tests) |
| **Held-out "gold" evals / human review during training** | detects the divergence | n/a - detection, not prevention |

In [ ]:
# The KL-regularised optimum for several betas: best TRUE score is at an intermediate drift, not at either extreme.
grid = np.logspace(-1.5, 1.5, 31); true_by_beta = [float(pi_star(R_PROXY, b) @ R_TRUE) for b in grid]
bi = int(np.argmax(true_by_beta)); print(f"best beta on this grid: {grid[bi]:.2f}  -> true score {true_by_beta[bi]:.2f}  (pi_ref scores {PI_REF @ R_TRUE:.2f}; beta->0 scores {true_by_beta[0]:.2f})")
assert 0 < bi < len(grid) - 1 and true_by_beta[bi] > PI_REF @ R_TRUE + 0.1 and true_by_beta[bi] > true_by_beta[0] + 0.5
# In practice you do not know R_TRUE; you choose beta / stopping point using a gold eval set, which is why A04's discipline matters here.

## 8 · Exercises
**Practice (30 min)**
1. In `train_dpo`, change `BETA` to 0.1 and to 5. How does the final policy's KL to `PI_REF` change? Explain using $\pi^*\propto\pi_{ref}e^{r/\beta}$ and assert the ordering.
2. Write `bt_prob(ra, rb)` and a test that `bt_prob(r+c, r'+c)` is invariant to the shift `c` (identifiability).

**Stretch (60 min)**
3. Add **label noise** (flip 20% of pair labels) to `sample_pairs` and plot KL(pi* || DPO) against the flip rate. What does DPO converge to as noise grows?
4. Implement **IPO-style** or a **length-normalised** variant of the DPO margin and show how it changes behaviour when `verbose` answers are systematically preferred.

**Challenge (2 h)**
5. Extend the GRPO toy with a **KL penalty to a reference policy** and a **clipped ratio** with 4 inner updates per batch. Show a setting where clipping prevents a collapse that unclipped inner updates cause, and report it over 10 seeds.

In [ ]:
# Practice 2 starter
def bt_prob(r_a, r_b):
    # TODO: Bradley-Terry probability that A is preferred over B
    raise NotImplementedError
# assert abs(bt_prob(1.0, 0.0) - sigmoid(1.0)) < 1e-12
# assert abs(bt_prob(1.0 + 7, 0.0 + 7) - bt_prob(1.0, 0.0)) < 1e-12   # only differences matter

## ✅ Checkpoint
1. Why can a Bradley-Terry reward model not tell you whether an answer is "good" in absolute terms?
2. Derive in two lines why the partition function $Z$ disappears from the DPO loss.
3. Your GRPO run's loss curve is flat on a dataset where the model already solves 100% of prompts, and also flat on prompts it solves 0%. Same bug? What do you do?
4. Your RLHF'd model scores higher and higher on the reward model but users complain it is wordy and flattering. Diagnose and give two fixes.

<details><summary>Answers</summary>

1. BT depends only on differences $r(y_w)-r(y_l)$, so $r$ is identified only up to an additive constant (and scale is fixed by the noise model). Compare responses to each other; do not threshold raw scores across prompts.
2. $r=\beta\log\frac{\pi^*}{\pi_{ref}}+\beta\log Z(x)$. In $r(y_w)-r(y_l)$ both responses share the same prompt $x$, so the $\beta\log Z(x)$ terms cancel exactly, leaving only log-ratios of the policy that we can compute.
3. Not a bug: groups with identical rewards have zero advantage, so no gradient. Filter or reweight the data toward prompts with intermediate pass rates (curriculum/online filtering), increase $G$ or temperature for diversity, or add partial-credit rewards.
4. Reward hacking of a biased proxy (annotators/RM favour length and flattery). Fixes: increase $\beta$ or stop earlier on a gold eval; rebalance/augment preference data (length-controlled comparisons, adversarial pairs); add explicit penalties or a length-normalised objective; use verifiable rewards where available.
</details>

## 📌 Takeaways
- Preferences are **comparisons**; Bradley-Terry turns them into a reward (up to a constant), and the model learns *rater* preferences - bias included.
- RLHF objective $=\mathbb E[r]-\beta\,\mathrm{KL}(\pi\|\pi_{ref})$, optimum $\pi^*\propto\pi_{ref}e^{r/\beta}$; $\beta$ trades reward against drift.
- **DPO** substitutes the optimum's implicit reward into BT: $\mathcal L=-\log\sigma(\beta[\Delta_w-\Delta_l])$, gradient $-\beta\sigma(-m)(\nabla\log\pi_w-\nabla\log\pi_l)$; no reward model, but no free lunch on data coverage.
- **GRPO/RLVR** uses group-relative advantages from a verifier: cheap baseline, no value net, no signal when a group is uniform.
- Optimising a proxy hard enough *will* find its flaws; keep a gold eval and a leash (KL / early stopping).

## 📚 Further reading
- Bradley & Terry (1952), *Rank analysis of incomplete block designs: I. The method of paired comparisons*, Biometrika
- Ouyang et al., *Training language models to follow instructions with human feedback* (2022), arXiv:2203.02155
- Schulman et al., *Proximal Policy Optimization Algorithms* (2017), arXiv:1707.06347
- Rafailov et al., *Direct Preference Optimization: Your Language Model is Secretly a Reward Model* (2023), arXiv:2305.18290
- Shao et al., *DeepSeekMath* (2024), arXiv:2402.03300 (introduces GRPO)
- Gao, Schulman & Hilton, *Scaling Laws for Reward Model Overoptimization* (2022), arXiv:2210.10760

**Next -> A07 · Reasoning Models & Test-Time Compute** (once a model is trained with verifiable rewards, how do we *spend* compute at inference time?)